In [3]:
from pathlib import Path
import pandas as pd

RAW = Path("../data/raw/wta")

frames = []
for path in sorted(RAW.glob("wta_matches_[0-9][0-9][0-9][0-9].csv")):
    df = pd.read_csv(path, low_memory=False)
    df["source_year"] = int(path.stem[-4:])
    frames.append(df)

matches = pd.concat(frames, ignore_index=True)

print(matches.shape)
print(matches["source_year"].nunique(), "ficheros")
matches.dtypes

(160887, 50)
58 ficheros


tourney_id                str
tourney_name              str
surface                   str
draw_size              object
tourney_level             str
tourney_date            int64
match_num               int64
winner_id               int64
winner_seed            object
winner_entry           object
winner_name               str
winner_hand               str
winner_ht             float64
winner_ioc                str
winner_age            float64
loser_id                int64
loser_seed             object
loser_entry               str
loser_name                str
loser_hand                str
loser_ht              float64
loser_ioc                 str
loser_age             float64
score                     str
best_of                 int64
round                     str
minutes               float64
w_ace                 float64
w_df                  float64
w_svpt                float64
w_1stIn               float64
w_1stWon              float64
w_2ndWon              float64
w_SvGms   

In [4]:
print(list(matches.columns))

check = ["surface", "tourney_level", "winner_rank", "loser_rank",
         "winner_ht", "loser_ht", "winner_age", "loser_age",
         "winner_hand", "loser_hand", "minutes", "w_ace", "w_svpt"]
check = [c for c in check if c in matches.columns]

coverage = (matches.groupby("source_year")[check]
            .agg(lambda s: s.notna().mean() * 100)
            .round(0))
coverage.insert(0, "matches", matches.groupby("source_year").size())

coverage.loc[[1968, 1975, 1980, 1985, 1990, 1995, 2000, 2005, 2010, 2015, 2020, 2025]]

['tourney_id', 'tourney_name', 'surface', 'draw_size', 'tourney_level', 'tourney_date', 'match_num', 'winner_id', 'winner_seed', 'winner_entry', 'winner_name', 'winner_hand', 'winner_ht', 'winner_ioc', 'winner_age', 'loser_id', 'loser_seed', 'loser_entry', 'loser_name', 'loser_hand', 'loser_ht', 'loser_ioc', 'loser_age', 'score', 'best_of', 'round', 'minutes', 'w_ace', 'w_df', 'w_svpt', 'w_1stIn', 'w_1stWon', 'w_2ndWon', 'w_SvGms', 'w_bpSaved', 'w_bpFaced', 'l_ace', 'l_df', 'l_svpt', 'l_1stIn', 'l_1stWon', 'l_2ndWon', 'l_SvGms', 'l_bpSaved', 'l_bpFaced', 'winner_rank', 'winner_rank_points', 'loser_rank', 'loser_rank_points', 'source_year']


,matches,surface,tourney_level,winner_rank,loser_rank,winner_ht,loser_ht,winner_age,loser_age,winner_hand,loser_hand,minutes,w_ace,w_svpt
source_year,,,,,,,,,,,,,,
1968,3068,97.0,100.0,0.0,0.0,30.0,13.0,92.0,72.0,100.0,100.0,0.0,0.0,0.0
1975,2703,88.0,100.0,0.0,0.0,51.0,33.0,95.0,86.0,100.0,100.0,0.0,0.0,0.0
1980,4032,85.0,100.0,0.0,0.0,48.0,33.0,91.0,80.0,100.0,100.0,0.0,0.0,0.0
1985,2705,93.0,100.0,38.0,24.0,60.0,45.0,100.0,99.0,100.0,100.0,0.0,0.0,0.0
1990,2728,100.0,100.0,98.0,97.0,67.0,50.0,100.0,100.0,100.0,100.0,0.0,0.0,0.0
1995,2529,100.0,100.0,96.0,93.0,79.0,69.0,100.0,100.0,100.0,100.0,0.0,0.0,0.0
2000,2893,100.0,100.0,94.0,90.0,86.0,79.0,100.0,100.0,100.0,100.0,0.0,0.0,0.0
2005,2843,100.0,100.0,98.0,96.0,95.0,90.0,100.0,100.0,100.0,100.0,0.0,19.0,19.0
2010,2781,100.0,100.0,98.0,95.0,96.0,92.0,100.0,100.0,100.0,100.0,0.0,78.0,78.0


In [5]:
full = (matches.groupby("source_year")[check]
        .agg(lambda s: s.notna().mean() * 100))

summary = pd.DataFrame({
    f"first year >= {t}%": full.apply(lambda col: col[col >= t].index.min())
    for t in (1, 50, 90)
})
display(summary)

for col in ["tourney_level", "surface", "winner_hand"]:
    print(matches[col].value_counts(dropna=False).to_string(), "\n")

,first year >= 1%,first year >= 50%,first year >= 90%
surface,1968,1968,1968
tourney_level,1968,1968,1968
winner_rank,1984,1984,1987
loser_rank,1984,1984,1987
winner_ht,1968,1973,2003
loser_ht,1968,1979,2005
winner_age,1968,1968,1968
loser_age,1968,1968,1976
winner_hand,1968,1968,1968
loser_hand,1968,1968,1968


tourney_level
W       70121
G       25753
I       15521
D       12224
P       10965
PM       5133
T1       4778
T3       4562
T2       4329
T4       2912
CC       1913
T5        968
O         672
F         533
E         282
50+H      184
35+H       31
J           6 

surface
Hard      67835
Clay      50458
Grass     23560
Carpet    14128
NaN        4875
clay         31 

winner_hand
R      120021
U       27704
L       13161
NaN         1 



In [6]:
from pathlib import Path
print(Path("../data/raw/wta/UPSTREAM_README.md").read_text(encoding="utf-8"))

## WTA Tennis Rankings, Results, and Stats

This repo contains my master WTA player file, historical rankings, and an extensive database of match results.

The player file columns are: player_id, first_name, last_name, hand, birth_date, country_code.

The columns for the ranking files are: ranking_date, ranking, player_id, ranking_points, tours. 'Tours' is WTA jargon for certain types of tournaments played, and that column is not anywhere near complete. 

Match result files (e.g. 'wta_matches_2014.csv') contain all tour-level singles matches for that season. To make them easier to use, I've included a fair bit of redundancy with the biographical and ranking files: each row contains several columns of biographical information, along with ranking and ranking points, for both players.

ITF, tour-level qualifying, and some ITF-level qualifying match results are stored in a separate yearly file (e.g. 'wta_matches_qual_itf_2014.csv').

Most of the columns in the results files are self-explan

In [7]:
lv = matches.groupby("tourney_level").agg(
    matches=("tourney_id", "size"),
    tournaments=("tourney_id", "nunique"),
    first_year=("source_year", "min"),
    last_year=("source_year", "max"),
    median_winner_rank=("winner_rank", "median"),
)
lv["examples"] = matches.groupby("tourney_level")["tourney_name"].agg(
    lambda s: ", ".join(s.drop_duplicates().sample(min(3, s.nunique()), random_state=1))
)
lv.sort_values("matches", ascending=False)

,matches,tournaments,first_year,last_year,median_winner_rank,examples
tourney_level,,,,,,
W,70121,3056,1968,2024,41.0,"Taipei, Maria Lankowitz, Landskron"
G,25753,231,1968,2025,29.0,"Us Open, Wimbledon, Roland Garros"
I,15521,496,2009,2025,70.0,"Ningbo, Cleveland, Chennai"
D,12224,5535,1968,2025,134.0,"Fed Cup WG F: USA vs ITA, Fed Cup G2 RRB: URU ..."
P,10965,277,2009,2025,27.0,"San Jose, Paris, Adelaide 2"
PM,5133,65,2009,2025,29.0,"Indian Wells, Miami, Cincinnati"
T1,4778,86,1985,2008,26.0,"Rome, Indian Wells, Zurich"
T3,4562,146,2000,2008,59.0,"Vienna, Oklahoma City, Bol"
T2,4329,142,2000,2008,22.0,"Luxembourg, Nice, Antwerp"


In [8]:
tiers = ["G", "I", "P", "PM", "T1", "T2", "T3", "T4", "T5", "F"]
matches["level_group"] = matches["tourney_level"].map(
    lambda l: "tour tiers" if l in tiers else ("W" if l == "W" else "other")
)

years = [1968, 1975, 1980, 1985, 1990, 1995, 2000, 2005, 2008, 2009, 2010, 2015, 2020, 2024]
display(pd.crosstab(matches["source_year"], matches["level_group"]).loc[years])

w = matches[matches["tourney_level"] == "W"]
for y in (1975, 2000, 2015, 2024):
    one = w[w["source_year"] == y].drop_duplicates("tourney_id")
    print(y, "-", len(one), "torneos W")
    print(one[["tourney_name", "draw_size"]].sample(min(6, len(one)), random_state=1).to_string(index=False), "\n")

level_group,W,other,tour tiers
source_year,,,
1968,2685,69,314
1975,2295,108,300
1980,3613,111,308
1985,2010,124,571
1990,2068,152,508
1995,1638,328,563
2000,0,599,2294
2005,0,294,2549
2008,0,387,2404


1975 - 109 torneos W
tourney_name draw_size
     Torquay       NaN
   Haverford       NaN
     Brumana       NaN
      Moscow       NaN
Buenos Aires       NaN
       Tours       NaN 

2000 - 0 torneos W
Empty DataFrame
Columns: [tourney_name, draw_size]
Index: [] 

2015 - 1 torneos W
tourney_name draw_size
  WTA Finals         8 

2024 - 1 torneos W
    tourney_name draw_size
Buenos Aires 125        32 



In [9]:
tour_levels = ["G", "W", "I", "P", "PM", "T1", "T2", "T3", "T4", "T5", "F"]
tour = matches[matches["tourney_level"].isin(tour_levels)].copy()
print(len(tour), "partidos de circuito principal")

dup_matches = tour.duplicated(["tourney_id", "match_num"]).sum()
same_player = (tour["winner_id"] == tour["loser_id"]).sum()
date = pd.to_datetime(tour["tourney_date"].astype(str), format="%Y%m%d", errors="coerce")
bad_dates = date.isna().sum()
year_mismatch = (date.dt.year != tour["source_year"]).sum()
nulls = tour[["score", "round", "winner_id", "loser_id"]].isna().sum()

print(f"duplicados (torneo+partido): {dup_matches}")
print(f"misma jugadora gana y pierde: {same_player}")
print(f"fechas ilegibles: {bad_dates}")
print(f"fecha de otro año que el fichero: {year_mismatch}")
print(nulls.to_string())

145575 partidos de circuito principal
duplicados (torneo+partido): 16
misma jugadora gana y pierde: 1
fechas ilegibles: 0
fecha de otro año que el fichero: 1291
score        1
round        0
winner_id    0
loser_id     0


In [10]:
key = ["tourney_id", "match_num"]
cols = key + ["tourney_name", "winner_name", "loser_name", "score", "source_year"]

display(tour[tour.duplicated(key, keep=False)].sort_values(key)[cols])
display(tour[tour["winner_id"] == tour["loser_id"]][cols])

diff = date.dt.year - tour["source_year"]
print(diff[diff != 0].value_counts().to_string())
print(date[diff != 0].dt.month.value_counts().sort_index().to_string())

,tourney_id,match_num,tourney_name,winner_name,loser_name,score,source_year
4060,1969-1029,1,Barranquilla,Faye Urban,Beatriz Del Corral,12-10 1-2 RET,1969
4684,1969-1029,1,Barranquilla,Almut Sturm,Judith Salome,6-1 6-2,1969
14920,1973-W-SL-USA-01A-1973,1,US Open,Billie Jean King,Peggy Michel,6-2 6-3,1973
17466,1973-W-SL-USA-01A-1973,1,US Open,Kathy May,Fiorella Bonicelli,4-6 7-6 6-3,1973
14921,1973-W-SL-USA-01A-1973,2,US Open,Karen Krantzcke,Sharon Walsh Pete,6-2 6-2,1973
17467,1973-W-SL-USA-01A-1973,2,US Open,Isabel Fernandez De Soto,Lesley Charles,6-1 6-1,1973
14922,1973-W-SL-USA-01A-1973,3,US Open,Janice Metcalf,Kerry Harris,5-7 6-1 6-4,1973
17468,1973-W-SL-USA-01A-1973,3,US Open,Laurie Fleming,Linda Thomas,6-1 6-1,1973
14923,1973-W-SL-USA-01A-1973,4,US Open,Julie Heldman,Laurie Tenney,6-1 6-4,1973
17469,1973-W-SL-USA-01A-1973,4,US Open,Laura Dupont,Brenda Kirk,6-7 6-0 6-2,1973


,tourney_id,match_num,tourney_name,winner_name,loser_name,score,source_year
35442,1980-1040,17,Berkeley,Marcie Louie,Marcie Louie,6-2 4-4 RET,1980


-1    1291
tourney_date
12    1291


In [13]:
decades = ["80s", "90s", "00s", "10s", "20s", "current"]
rankings = pd.concat(
    [pd.read_csv(RAW / f"wta_rankings_{d}.csv") for d in decades],
    ignore_index=True,
)
print(rankings.shape)
print(rankings.dtypes.to_string())

rankings["date"] = pd.to_datetime(
    rankings["ranking_date"].astype(str), format="%Y%m%d", errors="coerce")
print(rankings["date"].min(), rankings["date"].max(), rankings["date"].isna().sum())
print("repetidos (fecha+jugadora):", rankings.duplicated(["ranking_date", "player"]).sum())

rankings.groupby(rankings["date"].dt.year)["date"].nunique().to_string()

(2154318, 5)
ranking_date      int64
rank              int64
player            int64
points          float64
tours           float64
1984-01-02 00:00:00 2026-06-08 00:00:00 0
repetidos (fecha+jugadora): 281


'date\n1984    53\n1985    52\n1986    52\n1987    52\n1988    52\n1989    52\n1990    53\n1991    52\n1992    52\n1993    52\n1994    52\n1995    52\n1996    53\n1997    52\n1998    52\n1999    52\n2000    53\n2001    53\n2002    53\n2003    53\n2004    53\n2005    53\n2006    53\n2007    53\n2008    53\n2009    53\n2010    53\n2011    53\n2012    54\n2013    51\n2014    52\n2015    52\n2016    51\n2017    44\n2018    47\n2019    45\n2020    32\n2021    52\n2022    50\n2023    51\n2024    52\n2025    52\n2026    23'

In [14]:
players = pd.read_csv(RAW / "wta_players.csv", low_memory=False)
print(players.shape)
print(players.dtypes.to_string())
display(players.head())

(70571, 8)
player_id        int64
name_first         str
name_last          str
hand               str
dob            float64
ioc                str
height         float64
wikidata_id        str


,player_id,name_first,name_last,hand,dob,ioc,height,wikidata_id
0,113190,Bobby,Riggs,U,NaN,USA,NaN,NaN
1,200000,X,X,U,19000000.0,UNK,NaN,NaN
2,200001,Martina,Hingis,R,19800930.0,SUI,170.0,Q134720
3,200002,Mirjana,Lucic,R,19820309.0,CRO,181.0,Q239686
4,200003,Justine,Henin,R,19820601.0,BEL,167.0,Q11682


In [15]:
key = ["ranking_date", "player"]
dups = rankings[rankings.duplicated(key, keep=False)].sort_values(key)

print("filas implicadas:", len(dups))
print("copias exactas (fecha, jugadora, rank, points):",
      rankings.duplicated(key + ["rank", "points"]).sum())
print(dups["ranking_date"].value_counts().head(10).to_string())
display(dups.head(10))

filas implicadas: 562
copias exactas (fecha, jugadora, rank, points): 55
ranking_date
20260223    26
20240429    24
20251229    24
20251201    22
20260119    20
20040510    16
20240603    14
20250310    14
20260316    14
20231009    12


,ranking_date,rank,player,points,tours,date
646387,20040510,523,201469,23.0,15.0,2004-05-10
646388,20040510,486,201469,28.0,0.0,2004-05-10
649217,20040510,506,201478,25.0,24.0,2004-05-10
649218,20040510,485,201478,28.0,0.0,2004-05-10
663570,20040510,526,201525,23.0,6.0,2004-05-10
663571,20040510,495,201525,27.0,0.0,2004-05-10
712048,20040510,507,202048,25.0,NaN,2004-05-10
712049,20040510,493,202048,27.0,0.0,2004-05-10
727762,20040510,547,202288,21.0,NaN,2004-05-10
727763,20040510,490,202288,27.0,0.0,2004-05-10


In [16]:
conf = dups[~dups.duplicated(key + ["rank", "points"], keep=False)]
print("filas en conflicto:", len(conf), "| mejor rank implicado:", conf["rank"].min())
print("conflictos con rank <= 100:", (conf["rank"] <= 100).sum())

ids = pd.concat([tour["winner_id"], tour["loser_id"]]).unique()
pl = players[players["player_id"].isin(ids)]
print("jugadoras en partidos:", len(ids), "| encontradas en players:", len(pl))
print("dob informada:", round(pl["dob"].notna().mean() * 100), "% | altura:",
      round(pl["height"].notna().mean() * 100), "%")
print(pl["hand"].value_counts(dropna=False).to_string())
print("player_id repetidos en players:", players["player_id"].duplicated().sum())

filas en conflicto: 452 | mejor rank implicado: 221
conflictos con rank <= 100: 0
jugadoras en partidos: 5941 | encontradas en players: 5941
dob informada: 55 % | altura: 18 %
hand
U    4124
R    1619
L     198
player_id repetidos en players: 0


In [17]:
games = pd.concat([tour["winner_id"], tour["loser_id"]]).value_counts()
big = games[games >= 50].index
pb = players[players["player_id"].isin(big)]

print("jugadoras con >= 50 partidos:", len(pb))
print("dob:", round(pb["dob"].notna().mean() * 100), "% | altura:",
      round(pb["height"].notna().mean() * 100), "% | mano conocida:",
      round((pb["hand"] != "U").mean() * 100), "%")
print("sin país (nulo o UNK):", (pb["ioc"].isna() | (pb["ioc"] == "UNK")).sum())

jugadoras con >= 50 partidos: 1136
dob: 99 % | altura: 61 % | mano conocida: 73 %
sin país (nulo o UNK): 0


In [18]:
print(tour["round"].value_counts(dropna=False).to_string(), "\n")

tokens = tour["score"].fillna("").str.findall(r"[A-Za-z/]+").explode().value_counts()
print(tokens.to_string(), "\n")

names = tour["tourney_name"].str.strip()
spellings = names.groupby(names.str.lower()).nunique()
variants = names[names.str.lower().isin(spellings[spellings > 1].index)]
print("nombres con varias grafías:", (spellings > 1).sum())
print(variants.value_counts().sort_index().head(30).to_string(), "\n")

MATCH_KEY = ["tourney_id", "round", "winner_id", "loser_id", "score"]
print("duplicados reales con la clave de ATP:", tour.duplicated(MATCH_KEY).sum())

round
R32     48908
R16     29040
R64     23485
QF      15806
R128    14228
SF       8521
F        4617
RR        769
BR        201 

score
RET     2717
W/O     1235
UNK      160
DEF       27
nbsp      18
Oct        6
Sep        2
H          2
ABD        1
ab         1
ret        1
Aug        1 

nombres con varias grafías: 6
tourney_name
Cava Dei Tirreni       1
Cava dei Tirreni       1
Frinton-On-Sea       146
Frinton-on-Sea        38
Lee-On-Solent         89
Lee-on-Solent         72
S-Hertogenbosch       31
US Open             5830
Us Open              889
Vina Del Mar          31
Vina del Mar           5
s-Hertogenbosch       31 

duplicados reales con la clave de ATP: 38


In [19]:
common = {"RET", "W/O", "DEF"}
tokens = tour["score"].fillna("").str.findall(r"[A-Za-z/]+")
odd = tour[tokens.map(lambda ts: any(t not in common for t in ts))]

print(len(odd), "filas con texto raro en el marcador")
print(odd["score"].value_counts().head(40).to_string(), "\n")

dups = tour[tour.duplicated(MATCH_KEY, keep=False)].sort_values(MATCH_KEY)
print(len(dups), "filas implicadas en duplicados")
display(dups[MATCH_KEY + ["match_num", "source_year"]].head(16))

192 filas con texto raro en el marcador
score
UNK                   160
&nbsp;                 18
7-Oct                   2
7-Sep                   2
3-Oct                   1
6-Oct                   1
2-Oct                   1
5-Oct                   1
5-3 ABD                 1
5-4 ab                  1
2-1 ret                 1
Aug-00                  1
6-1 4-3 RET+H64         1
7-6(0) 0-2 RET+H61      1 

76 filas implicadas en duplicados


,tourney_id,round,winner_id,loser_id,score,match_num,source_year
17019,1973-1098,R32,200208,201162,7-5 6-4,29,1973
17022,1973-1098,R32,200208,201162,7-5 6-4,32,1973
17020,1973-1098,R32,200275,200286,6-4 6-2,30,1973
17023,1973-1098,R32,200275,200286,6-4 6-2,33,1973
40317,1981-1099,RR,200237,200365,6-4 3-6 6-1,7,1981
40327,1981-1099,RR,200237,200365,6-4 3-6 6-1,17,1981
40318,1981-1099,RR,200237,201018,1-6 6-2 6-3,8,1981
40328,1981-1099,RR,200237,201018,1-6 6-2 6-3,18,1981
40313,1981-1099,RR,200357,200419,4-6 7-5 6-3,3,1981
40323,1981-1099,RR,200357,200419,4-6 7-5 6-3,13,1981


In [20]:
names = tour["tourney_name"].str.strip()
spellings = names.groupby(names.str.lower()).nunique()
multi = spellings[spellings > 1].index
variants = names[names.str.lower().isin(multi)]

print("nombres con varias grafías:", len(multi))
print(variants.value_counts().sort_index().head(40).to_string())

nombres con varias grafías: 6
tourney_name
Cava Dei Tirreni       1
Cava dei Tirreni       1
Frinton-On-Sea       146
Frinton-on-Sea        38
Lee-On-Solent         89
Lee-on-Solent         72
S-Hertogenbosch       31
US Open             5830
Us Open              889
Vina Del Mar          31
Vina del Mar           5
s-Hertogenbosch       31


In [22]:
ids_clean = set(pd.read_csv("../data/interim/wta_players_clean.csv",
                            usecols=["id"], dtype=str)["id"])

rk = rankings[rankings["rank"] <= 50].copy()
rk["id"] = rk["player"].astype(str)
miss = rk[~rk["id"].isin(ids_clean)]
print(len(miss), "filas del top 50 de jugadoras que faltan;",
      miss["id"].nunique(), "jugadoras")

top = (miss.groupby("id")
       .agg(rows=("rank", "size"), best=("rank", "min"),
            first=("date", "min"), last=("date", "max"))
       .sort_values("rows", ascending=False).head(15).reset_index())

raw_players = pd.read_csv(RAW / "wta_players.csv", dtype={"player_id": str}, low_memory=False)
top = top.merge(raw_players[["player_id", "name_first", "name_last"]],
                left_on="id", right_on="player_id", how="left")

in_raw_matches = set(matches["winner_id"].astype(str)) | set(matches["loser_id"].astype(str))
top["in_raw_matches"] = top["id"].isin(in_raw_matches)
display(top)

4 filas del top 50 de jugadoras que faltan; 2 jugadoras


,id,rows,best,first,last,player_id,name_first,name_last,in_raw_matches
0,213027,2,18,1986-02-03,1986-02-10,213027,Mary,Innes,False
1,213029,2,46,1986-05-26,1986-06-02,213029,Chris Lynne,Kuhlke,False


In [23]:
clean = pd.read_csv("../data/interim/wta_rankings_clean.csv",
                    parse_dates=["date"], dtype={"id": str})
names = pd.read_csv("../data/interim/wta_players_clean.csv",
                    usecols=["id", "name"], dtype=str)

ones = clean[clean["rank"] == 1].groupby("date").size()
multi = ones[ones > 1].index
all_dates = clean["date"].drop_duplicates()
none = all_dates[~all_dates.isin(ones.index)]

print("fechas sin No. 1:", none.dt.date.tolist())
print("más de un No. 1, por año:")
print(multi.year.value_counts().sort_index().to_string())
display(clean[clean["date"].isin(multi[:3]) & (clean["rank"] == 1)]
        .merge(names, on="id", how="left"))

fechas sin No. 1: [datetime.date(1984, 2, 27), datetime.date(1985, 8, 19), datetime.date(1985, 8, 26), datetime.date(2004, 5, 10), datetime.date(2022, 5, 30), datetime.date(2023, 1, 23)]
más de un No. 1, por año:
date
1995    20
1996    45


,date,id,rank,points,name
0,1995-08-14,200414,1,360.0,Steffi Graf
1,1995-08-14,200652,1,NaN,Monica Seles
2,1995-08-21,200414,1,333.0,Steffi Graf
3,1995-08-21,200652,1,NaN,Monica Seles
4,1995-08-28,200414,1,333.0,Steffi Graf
5,1995-08-28,200652,1,NaN,Monica Seles


In [24]:
bad = pd.to_datetime(["1984-02-27", "1985-08-19", "1985-08-26",
                      "2004-05-10", "2022-05-30", "2023-01-23"])
raw_bad = rankings[rankings["date"].isin(bad)].sort_values(["date", "rank"])

print(raw_bad.groupby("date")["rank"].agg(["min", "max", "size"]).to_string())
display(raw_bad.groupby("date").head(3)[["date", "rank", "player", "points"]])

            min   max  size
date                       
1984-02-27   18    18     1
1985-08-19   64    73     2
1985-08-26   64    73     2
2004-05-10  101   600   500
2022-05-30  201  1560  1100
2023-01-23  201  1290  1000


,date,rank,player,points
885,1984-02-27,18,200419,NaN
7664,1985-08-19,64,200630,NaN
7665,1985-08-19,73,200407,NaN
7666,1985-08-26,64,200630,NaN
7667,1985-08-26,73,200407,NaN
513618,2004-05-10,101,200045,358.0
615967,2004-05-10,102,201390,351.0
515530,2004-05-10,103,200050,338.0
1857262,2022-05-30,201,206430,333.0
1857263,2022-05-30,202,213672,332.0


In [27]:
import pandas as pd

atp = pd.read_csv("../data/interim/matches_clean.csv", nrows=0).columns
wta = pd.read_csv("../data/interim/wta_matches_clean.csv", nrows=0).columns

print("Solo en ATP:", sorted(set(atp) - set(wta)))
print("Solo en WTA:", sorted(set(wta) - set(atp)))
print("Mismo orden en las comunes:", [c for c in atp if c in wta] == [c for c in wta if c in atp])

Solo en ATP: ['indoor']
Solo en WTA: []
Mismo orden en las comunes: False


In [28]:
import pandas as pd

m = pd.read_csv("../data/interim/wta_matches_clean.csv",
                usecols=["tourney_id", "tourney_name", "tourney_level", "season", "outcome"])
m = m[m["outcome"].isin(["completed", "retired"])]

per_t = (m[m["season"].isin([1968, 1972, 1985, 2015])]
         .groupby(["season", "tourney_id"]).size()
         .rename("matches").reset_index())

print(per_t.groupby("season")["matches"].describe()[["count", "min", "50%", "max"]])
print()
print(m[m["season"] == 1968].groupby("tourney_level").size())

        count   min   50%    max
season                          
1968    236.0   1.0   7.0   94.0
1972    183.0   1.0  10.0  117.0
1985     63.0   3.0  31.0  127.0
2015     59.0  15.0  31.0  127.0

tourney_level
G     278
W    2643
dtype: int64


In [29]:
import pandas as pd

cols = ["tourney_id", "tourney_name", "season", "round_clean", "winner_name", "loser_name", "score", "outcome", "date"]
for name, f in [("ATP", "../data/interim/matches_clean.csv"), ("WTA", "../data/interim/wta_matches_clean.csv")]:
    m = pd.read_csv(f, usecols=cols, low_memory=False)
    finals = m[m["round_clean"] == "F"]
    many = finals.groupby("tourney_id").size()
    ids = many[many > 1].index
    print(name)
    print(finals[finals["tourney_id"].isin(ids)].sort_values(["tourney_id", "date"]).to_string(index=False))
    print()

ATP
tourney_id tourney_name     winner_name         loser_name   score  season   outcome round_clean       date
  1978-615  Nations Cup   Jose Higueras      John Newcombe 6-2 6-3    1978 completed           F 1978-05-08
  1978-615  Nations Cup  Manuel Orantes          Phil Dent 6-3 6-4    1978 completed           F 1978-05-08
  1979-615  Nations Cup  John Alexander Corrado Barazzutti 6-2 6-0    1979 completed           F 1979-05-07
  1979-615  Nations Cup Adriano Panatta          Phil Dent 6-3 6-3    1979 completed           F 1979-05-07
  1980-615  Nations Cup Jose-Luis Clerc    Adriano Panatta 7-6 6-3    1980 completed           F 1980-05-05
  1980-615  Nations Cup Guillermo Vilas Corrado Barazzutti 6-3 6-2    1980 completed           F 1980-05-05
  1981-615  Nations Cup  Peter McNamara         Ivan Lendl 6-3 6-4    1981 completed           F 1981-05-04
  1981-615  Nations Cup      Tomas Smid       Paul McNamee 6-4 7-6    1981 completed           F 1981-05-04

WTA
tourney_id tourney_

In [30]:
import pandas as pd

pattern = r"Cup|Team|Laver|Hopman|Davis|Fed |Olymp|Nations|Atp Cup|ATP Cup|Hopman"
cols = ["tourney_id", "tourney_name", "level_group", "season"]

for name, f in [("ATP", "../data/interim/matches_clean.csv"), ("WTA", "../data/interim/wta_matches_clean.csv")]:
    m = pd.read_csv(f, usecols=cols, low_memory=False)
    hit = m[m["tourney_name"].str.contains(pattern, case=False, regex=True, na=False)]
    summary = (hit.groupby(["tourney_name", "level_group"])
               .agg(matches=("tourney_id", "size"), editions=("tourney_id", "nunique"),
                    first=("season", "min"), last=("season", "max"))
               .reset_index())
    print(name)
    print(summary.to_string(index=False))
    print()

ATP
       tourney_name            level_group  matches  editions  first  last
     Grand Slam Cup          Special event      142        10   1990  1999
        Masters Cup            Tour Finals      120         8   2001  2008
        Nations Cup Tour event (pre-tiers)      134         5   1975  1981
     Seoul Olympics Tour event (pre-tiers)       17         1   1988  1988
 Tennis Masters Cup            Tour Finals       15         1   2000  2000
WCT Aetna World Cup Tour event (pre-tiers)       49        10   1970  1979
  WCT Challenge Cup Tour event (pre-tiers)       91         7   1976  1980
      WCT World Cup Tour event (pre-tiers)       16         1   1968  1968

WTA
                      tourney_name level_group  matches  editions  first  last
                    Grand Slam Cup  Tour event       14         2   1998  1999
                       Kremlin Cup  Tour event      110         4   1996  1999
 Sao Paulo Osorio Cup F CHI vs COL  Tour event        4         1   1970  1970


In [31]:
import pandas as pd

cols = ["tourney_id", "tourney_name", "level_group", "season", "round_clean",
        "winner_name", "loser_name", "winner_ioc", "loser_ioc"]
atp = pd.read_csv("../data/interim/matches_clean.csv", usecols=cols, low_memory=False)
wta = pd.read_csv("../data/interim/wta_matches_clean.csv", usecols=cols, low_memory=False)

print("ATP Special event:")
print(atp[atp["level_group"] == "Special event"]
      .groupby(["tourney_name", "season"]).size().rename("matches").to_string())

for name, m in [("ATP", atp), ("WTA", wta)]:
    vs = m[m["tourney_name"].str.contains(r"\bvs\b", case=False, regex=True, na=False)]
    print(f"\n{name} names containing ' vs ':", vs["tourney_name"].nunique(), "names,", len(vs), "matches")

print("\nWCT Challenge Cup, sample:")
c = atp[atp["tourney_name"] == "WCT Challenge Cup"]
print(c.head(8)[["season", "round_clean", "winner_name", "winner_ioc", "loser_name", "loser_ioc"]].to_string(index=False))

ATP Special event:
tourney_name     season
Grand Slam Cup   1990      15
                 1991      15
                 1992      15
                 1993      15
                 1994      15
                 1995      15
                 1996      15
                 1997      15
                 1998      11
                 1999      11
Next Gen Finals  2017      16
                 2018      16
                 2019      15
                 2021      15
                 2022      15
                 2023      15
                 2024      15
                 2025      15

ATP names containing ' vs ': 0 names, 0 matches

WTA names containing ' vs ': 19 names, 46 matches

WCT Challenge Cup, sample:
 season round_clean  winner_name winner_ioc     loser_name loser_ioc
   1976          RR  Arthur Ashe        USA   Raul Ramirez       MEX
   1976          RR   Bjorn Borg        SWE  John Newcombe       AUS
   1976          RR    Rod Laver        AUS John Alexander       AUS
   1976      

In [32]:
import pandas as pd

wta = pd.read_csv("../data/interim/wta_matches_clean.csv",
                  usecols=["tourney_name", "season", "level_group"], low_memory=False)
vs = wta[wta["tourney_name"].str.contains(r"\bvs\b", case=False, regex=True, na=False)]
print(vs.groupby(["season", "tourney_name"]).size().rename("matches").to_string())

season  tourney_name                      
1968    Edinburgh ENG vs SCO                  2
        Perth AUS vs USA                      2
1970    Athens Balkan Games BUL vs GRE        2
        Athens Balkan Games BUL vs ROU        2
        Athens Balkan Games BUL vs TUR        2
        Athens Balkan Games BUL vs YUG        2
        Athens Balkan Games GRE vs ROU        2
        Athens Balkan Games GRE vs TUR        2
        Athens Balkan Games GRE vs YUG        2
        Athens Balkan Games ROU vs TUR        2
        Athens Balkan Games ROU vs YUG        2
        Athens Balkan Games TUR vs YUG        2
        Sao Paulo Osorio Cup F CHI vs COL     4
        Sao Paulo Osorio Cup QF BRA vs ECU    3
        Sao Paulo Osorio Cup QF CHI vs PER    2
        Sao Paulo Osorio Cup QF COL vs URU    2
        Sao Paulo Osorio Cup SF ARG vs COL    4
        Sao Paulo Osorio Cup SF CHI vs ECU    2
1972    Beckenham GB vs World                 5


In [33]:
import pandas as pd

for tour, players_file in [("atp", "players_clean.csv"), ("wta", "wta_players_clean.csv")]:
    titles = pd.read_csv(f"../data/processed/{tour}_titles.csv", dtype={"player_id": str})
    players = pd.read_csv(f"../data/interim/{players_file}", usecols=["id", "name"], dtype={"id": str})

    top = (titles.groupby("player_id")[["titles", "slams"]].sum()
           .sort_values(["titles", "slams"], ascending=False)
           .head(10)
           .reset_index()
           .merge(players, left_on="player_id", right_on="id", how="left"))
    print(tour.upper())
    print(top[["name", "titles", "slams"]].to_string(index=False))
    print()

ATP
          name  titles  slams
 Jimmy Connors     110      8
 Roger Federer     103     20
Novak Djokovic     100     24
    Ivan Lendl      95      8
  Rafael Nadal      91     22
  John McEnroe      78      7
     Rod Laver      72      5
    Bjorn Borg      66     11
  Ilie Nastase      65      2
  Pete Sampras      64     14

WTA
               name  titles  slams
Martina Navratilova     169     18
        Chris Evert     147     18
        Steffi Graf     106     22
     Margaret Court      95     11
   Evonne Goolagong      91      7
   Billie Jean King      82      9
    Serena Williams      72     23
      Virginia Wade      59      3
  Lindsay Davenport      54      3
     Helga Masthoff      54      0



In [34]:
import pandas as pd

m = pd.read_csv("../data/interim/wta_matches_clean.csv",
                usecols=["tourney_id", "tourney_name", "season", "round_clean", "winner_name", "level_group"],
                low_memory=False)

finals = m[m["round_clean"] == "F"]
evert = finals[finals["winner_name"] == "Chris Evert"].groupby("season").size()
print("Evert titles by season:")
print(evert.to_string())

with_final = set(finals["tourney_id"])
no_final = m[~m["tourney_id"].isin(with_final)].drop_duplicates("tourney_id")
print("\nWTA tournaments without a final, by decade:")
print((no_final["season"] // 10 * 10).value_counts().sort_index().to_string())

Evert titles by season:
season
1969     1
1970     3
1971     7
1972     4
1973    12
1974    12
1975    13
1976     9
1977    11
1978     7
1979     7
1980     8
1981     8
1982     9
1983     6
1984     6
1985     9
1986     6
1987     5
1988     4

WTA tournaments without a final, by decade:
season
1960     6
1970    15
1980     3
1990     2
2000     1
2020     1


In [35]:
import pandas as pd

m = pd.read_csv("../data/interim/wta_matches_clean.csv",
                usecols=["tourney_id", "tourney_name", "tourney_level", "season", "outcome", "round_clean"],
                low_memory=False)
m = m[m["outcome"].isin(["completed", "retired"])]

by_year = (m[m["season"].between(1977, 1984)].groupby("season")
           .agg(matches=("tourney_id", "size"), tournaments=("tourney_id", "nunique")))
by_year["per_tournament"] = (by_year["matches"] / by_year["tournaments"]).round(1)
print(by_year.to_string())

big = (m[m["season"] == 1980].groupby(["tourney_id", "tourney_name"]).size()
       .rename("matches").sort_values(ascending=False).head(12))
print()
print(big.to_string())

        matches  tournaments  per_tournament
season                                      
1977       2201          105            21.0
1978       2618           82            31.9
1979       2139           96            22.3
1980       3828          129            29.7
1981       3284          131            25.1
1982       2799          113            24.8
1983       2305           71            32.5
1984       2181           54            40.4

tourney_id              tourney_name   
1980-W-SL-USA-01A-1980  US Open            95
1980-W-SL-GBR-01A-1980  Wimbledon          95
1980-W-WT-USA-08A-1980  Detroit            64
1980-W-SL-FRA-01A-1980  Roland Garros      63
1980-W-WT-GBR-02A-1980  Eastbourne         63
1980-1037               Hampstead          62
1980-1051               Beckenham          60
1980-W-WT-USA-14A-1980  Hilton Head        56
1980-W-WT-CAN-02A-1980  Toronto            55
1980-W-WT-AUS-01A-1980  Sydney             55
1980-W-SL-AUS-01A-1980  Australian Open    55
198

In [36]:
import pandas as pd

m = pd.read_csv("../data/interim/wta_matches_clean.csv",
                usecols=["tourney_id", "tourney_name", "season", "round_clean",
                         "winner_id", "loser_id", "score", "outcome"],
                low_memory=False)
m = m[m["outcome"].isin(["completed", "retired"])].copy()

# 1. Matches per season by id style (new style contains "-W-")
m["id_style"] = m["tourney_id"].str.contains("-W-").map({True: "new", False: "old"})
print(m[m["season"].between(1975, 1986)].groupby(["season", "id_style"]).size().unstack(fill_value=0))

# 2. Same match under different tournament ids
key = ["season", "tourney_name", "round_clean", "winner_id", "loser_id", "score"]
dup = m.duplicated(key, keep=False)
print("\nRows that repeat across ids:", int(dup.sum()))
print(m[dup].groupby("season").size().sort_values(ascending=False).head(8).to_string())

# 3. Tournament names with more than one id in the same season
ids = m.groupby(["season", "tourney_name"])["tourney_id"].nunique()
print("\nName+season pairs with several ids:", int((ids > 1).sum()))
print(ids[ids > 1].groupby("season").size().sort_values(ascending=False).head(8).to_string())

id_style   new   old
season              
1975       275  2296
1976       284  1842
1977       887  1314
1978      1375  1243
1979      1312   827
1980      1907  1921
1981      1802  1482
1982      1863   936
1983      2142   163
1984      2107    74
1985      2318   254
1986      2333    77

Rows that repeat across ids: 0
Series([], )

Name+season pairs with several ids: 19
season
1977    2
1995    2
1968    1
1973    1
1970    1
1976    1
1986    1
1996    1


In [37]:
import pandas as pd

m = pd.read_csv("../data/interim/wta_matches_clean.csv",
                usecols=["tourney_id", "tourney_name", "season", "round_clean",
                         "winner_id", "loser_id", "score", "outcome"],
                low_memory=False)
m = m[m["outcome"].isin(["completed", "retired"])].copy()

key = ["season", "winner_id", "loser_id", "score"]
dup = m.duplicated(key, keep=False)
print("Rows sharing season, players and score:", int(dup.sum()))
print(m[dup].groupby("season").size().sort_values(ascending=False).head(8).to_string())

# does any of those pairs sit under two different tournament ids?
two_ids = m[dup].groupby(key)["tourney_id"].nunique()
print("\nSame match under two ids:", int((two_ids > 1).sum()))
print(m[dup].sort_values(key).head(12)[["season", "tourney_name", "tourney_id", "round_clean", "score"]].to_string(index=False))

Rows sharing season, players and score: 536
season
1973    28
1972    26
1970    24
1971    22
1981    22
1980    18
1968    16
1989    16

Same match under two ids: 261
 season      tourney_name             tourney_id round_clean   score
   1968          Istanbul              1968-1183          SF 6-1 6-0
   1968            Ankara              1968-1186          SF 6-1 6-0
   1968           US Open 1968-W-SL-USA-01A-1968         R32 6-0 6-1
   1968             Salem              1968-1015         R16 6-0 6-1
   1968          Istanbul              1968-1183          SF 6-2 6-4
   1968            Ankara              1968-1186          SF 6-2 6-4
   1968      Queen's Club              1968-1006         R32 6-0 6-2
   1968         Milwaukee              1968-1025         R32 6-0 6-2
   1968           Palermo              1968-1103         R32 6-4 6-2
   1968           Ortisei              1968-1200         R16 6-4 6-2
   1968 Manchester-On-Sea              1968-1029         R32 8-6 6-4
  

In [38]:
import pandas as pd

files = {"ATP": "../data/interim/matches_clean.csv", "WTA": "../data/interim/wta_matches_clean.csv"}
cols = ["season", "surface", "outcome", "w_svpt", "l_svpt", "minutes"]

for name, path in files.items():
    m = pd.read_csv(path, usecols=cols, low_memory=False)
    played = m[m["outcome"].isin(["completed", "retired"])]

    print(name, "- surface values:")
    print(played["surface"].value_counts(dropna=False).to_string())

    cov = (played.assign(serve=played["w_svpt"].notna() & played["l_svpt"].notna(),
                         mins=played["minutes"].notna())
           .groupby("season")[["serve", "mins"]].mean().mul(100).round(0))
    print("first season with serve stats on 50% of matches:", (cov["serve"] >= 50).idxmax())
    print("first season with serve stats on 90% of matches:", (cov["serve"] >= 90).idxmax())
    print(cov.loc[[1980, 1990, 1991, 1995, 2000, 2003, 2005, 2007, 2010, 2016, 2025]].to_string())
    print()

ATP - surface values:
surface
Hard      72473
Clay      62997
Grass     23454
Carpet    20255
first season with serve stats on 50% of matches: 1991
first season with serve stats on 90% of matches: 1991
        serve   mins
season              
1980      0.0    0.0
1990      0.0    0.0
1991    100.0  100.0
1995    100.0   93.0
2000    100.0   99.0
2003    100.0  100.0
2005    100.0  100.0
2007     99.0   99.0
2010    100.0  100.0
2016    100.0  100.0
2025    100.0   95.0

WTA - surface values:
surface
Hard       60739
Clay       43747
Grass      22178
Carpet     13351
Unknown     4142
first season with serve stats on 50% of matches: 2006
first season with serve stats on 90% of matches: 2014
        serve   mins
season              
1980      0.0    0.0
1990      0.0    0.0
1991      0.0    0.0
1995      0.0    0.0
2000      0.0    0.0
2003     15.0    0.0
2005     22.0    0.0
2007     77.0    0.0
2010     89.0    0.0
2016    100.0  100.0
2025    100.0   94.0



In [39]:
import pandas as pd

m = pd.read_csv("../data/interim/wta_matches_clean.csv",
                usecols=["season", "surface", "outcome", "w_svpt", "l_svpt", "minutes"], low_memory=False)
played = m[m["outcome"].isin(["completed", "retired"])]

print("surfaces:", played["surface"].value_counts(dropna=False).to_dict())

cov = (played.assign(serve=played["w_svpt"].notna() & played["l_svpt"].notna(),
                     mins=played["minutes"].notna())
       .groupby("season")[["serve", "mins"]].mean().mul(100).round(0))
print("serve >= 50% from:", (cov["serve"] >= 50).idxmax(), "| >= 90% from:", (cov["serve"] >= 90).idxmax())
print(cov.loc[2002:2017].T.to_string())

surfaces: {'Hard': 60739, 'Clay': 43747, 'Grass': 22178, 'Carpet': 13351, 'Unknown': 4142}
serve >= 50% from: 2006 | >= 90% from: 2014
season  2002  2003  2004  2005  2006  2007  2008  2009  2010  2011  2012  2013  2014  2015   2016   2017
serve    0.0  15.0  17.0  22.0  54.0  77.0  82.0  88.0  89.0  88.0  89.0  88.0  90.0  78.0  100.0  100.0
mins     0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  100.0   99.0


## Cleaning rules

The WTA match files are cleaned by `src/clean_wta_data.py`, which reads `data/raw/wta/wta_matches_<season>.csv` and writes `data/interim/wta_matches_clean.csv`. Every rule below comes from what this notebook showed; the script prints how many rows each one touches and stops if the result breaks a rule the dashboard relies on.

| # | Step | Rule | Why |
|---|------|------|-----|
| 1 | Seasons | Keep 1968 to 2025. | The 2026 file is a partial season. |
| 2 | Tournament names | Trim spaces; spellings that differ only in casing are unified to the most frequent one; five names are fixed by hand (`Vina del Mar`, `Cava dei Tirreni`, `Frinton-on-Sea`, `Lee-on-Solent`, `s-Hertogenbosch`). | The same event was written in several ways. The five fixes are cases where the most frequent spelling would be wrong. |
| 3 | Scope: level | Keep the main-tour levels only: `G`, `W`, `T1`–`T5`, `PM`, `P`, `I`, `F`. Drop Fed Cup (`D`), Olympics (`O`), ITF circuits (`CC`, `E`, `50+H`, `35+H`) and juniors (`J`). | The dashboard is about the individual professional tour. |
| 4 | Scope: team events | Drop rows whose tournament name contains `vs` (ties between countries, e.g. "Athens Balkan Games BUL vs GRE") and the United Cup. | These are team events that sit inside the main-tour levels. |
| 5 | Scores | Set to missing the scores that carry no information: `UNK`, `&nbsp;` and scores a spreadsheet turned into a date (`7-Oct`, `Aug-00`). The match itself is kept. | A bad score must not remove a match that was played. |
| 6 | Outcome | `walkover` if the score contains `W/O`; `retired` if it contains `RET`, `DEF`, `ABD` or `AB`; otherwise `completed`. Match length is cleared on walkovers. | A match that was never played cannot have a duration. |
| 7 | Level fix | The 2015 WTA Finals is recoded from `W` to `F`. | An 8-player finals coded as a normal event. |
| 8 | Level group | `Grand Slam`, `Tour Finals`, or `Tour event` for everything else. | The WTA tiers changed names over time (`W`, `T1`–`T5`, `PM`/`P`/`I`); one group per era is not needed here. |
| 9 | Rounds | `round_clean` keeps the source label; `round_order` numbers the knockout rounds R128 to F. `RR` (round robin) and `BR` (bronze) are kept but have no bracket position. | They are not part of a knockout draw. |
| 10 | Surface | Missing surfaces become `Unknown`. | Keeping the match is better than dropping it; the dashboard leaves `Unknown` out of the surface charts. |
| 11 | Same player | Rows where winner and loser have the same id are dropped. | A data-entry error; there is no way to tell which player was meant. |
| 12 | Duplicates | A match is a duplicate when tournament, round, winner, loser and score are all equal; only the first row is kept. | The same match was entered twice. |
| 13 | Date | `tourney_date` (e.g. `19750406`) becomes a real date. | |
| 14 | Final checks | Seasons within range, main-tour levels only, valid outcomes, a level group and a surface on every row, no same-player rows, no duplicates. | The script fails loudly instead of writing a table that breaks these rules. |

### What differs from the ATP cleaning
- The WTA has only three level groups (`Grand Slam`, `Tour Finals`, `Tour event`), because the source has no Masters / 500 / 250 ladder.
- Rows with the same winner and loser id are dropped. In the ATP data they could be repaired from the player's name; here they cannot.
- The WTA needs the scope rules for team events by name (`vs`, United Cup), because the pairing of countries is part of the tournament name.
- Score cleaning is WTA-only: the spreadsheet-date and `&nbsp;` scores do not appear in the ATP files.

### Limits of the source that the dashboard must respect
- The archive is a mirror of Jeff Sackmann's WTA data (the original repositories were removed). Early seasons are incomplete: the 1970s and early 1980s have missing titles and partial draws.
- Match rankings are reliable from 1986 (the ranking file starts in 1984, but 1985 has almost no rankings in the match rows).
- Serve statistics are reliable from 2006; height from 1993 and handedness from 1992. Before those years fewer than half of the matches carry the data.
- Surface is unknown for a large share of the matches before 1986.
- Country, height and handedness describe the player, not the match: one value per player, the one in the source.